# Per-hunk context selector: training and offline evaluation (Colab)

This notebook trains and evaluates a **context selector** on the offline per-hunk
outcome table (`aacr_hunk_15%`: 28 PRs, 20 repositories, 397 diff hunks,
190 reference comments, ten context levels `H01_min` to `H10_all`).

**What the selector does.** For each diff hunk it picks one context level for the
frozen reviewer (Qwen3.8-27B, one call per hunk). Every policy is scored by **lookup**
in the saved table: no reviewer or judge is called here, so a CPU runtime is enough.

**How a policy is scored** (the same rule as `docs/per_hunk_review_test.md`, Sections 19-20):
a PR's covered references are the union over its hunks of the references covered at
the level chosen for each hunk. Reported per policy:
- recall: covered references out of 190;
- severity-weighted score (weights 5 / 2 / 0.5 for blocking / should fix / nit,
  `docs/review_score_design.md`), plus the other four weightings;
- reviewer input tokens, as a share of "always `H03_file`";
- recall on human-written and on AI-written references, reported separately.

**Rules this notebook keeps** (roadmap Sections 9.3-9.4, the professor meeting):
- Features are **input side only**: diff size, trace counts, token counts per level,
  file path, PR text. Nothing from the references (category, context label, author,
  severity), the reviews or the judge enters a feature. A guard cell checks this.
- Cross-validation is **grouped by repository** (all hunks of a PR, and all PRs of a
  repository, stay in one fold).
- Every learned selector is compared with fixed levels, random choices at the same
  token budget, a rule with no learning, and the hindsight oracle (an upper bound).

**Limits to keep in mind.** This is 15% of the benchmark and **one** reviewer run.
On the five PRs with three runs, the run-to-run noise was about as large as some level
differences (per-hunk doc Section 17). The judge and the severity labeller are the
reviewer's own model and are not validated. Treat every number here as a pilot.

## 1. Setup

The notebook needs three files. Put them in one folder on Google Drive (about 2.4 MB)
and set `DATA_PATH` to that folder:
- `hunk_outcomes.json` (outcomes per hunk and level, reference tiers);
- `context_levels_slim.jsonl` (input-side features and the `H01_min` text; the full
  `context_levels.jsonl`, 122 MB, also works);
- `coverage.csv` (used only to check the outcomes; optional).

The files are found by name anywhere under `DATA_PATH`, so the original
`aacr_hunk_15%` folder or its zip also works.

In [ ]:
import os, sys, json, hashlib, zipfile, time, math, warnings, collections
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Folder (or zip) that holds the three files. Change this to where you put them on Drive.
DATA_PATH = os.environ.get(
    "SELECTOR_DATA", "/content/drive/MyDrive/CS536/selector_data")
# Results (CSVs, figures, run manifest) are written here.
OUT_DIR = Path(os.environ.get(
    "SELECTOR_OUT", "/content/drive/MyDrive/CS536/selector_results" if IN_COLAB else "selector_results"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_inputs(path):
    """Find the three input files by name under a folder (a zip is unzipped first)."""
    p = Path(path)
    if p.is_file() and p.suffix == ".zip":
        dest = Path("/content/selector_data_unzipped" if IN_COLAB else p.with_suffix(""))
        if not dest.exists():
            print(f"Unzipping {p} -> {dest}")
            with zipfile.ZipFile(p) as z:
                z.extractall(dest)
        p = dest
    if not p.is_dir():
        raise FileNotFoundError(f"DATA_PATH does not exist: {p}")
    def first(*names):
        for name in names:
            hits = sorted(h for h in p.rglob(name) if "__MACOSX" not in h.parts and not h.name.startswith("._"))
            if hits:
                return hits[0]
        return None
    files = {"outcomes": first("hunk_outcomes.json"),
             "contexts": first("context_levels_slim.jsonl", "context_levels.jsonl"),
             "coverage": first("coverage.csv")}
    for key in ("outcomes", "contexts"):
        if files[key] is None:
            raise FileNotFoundError(f"No {key} file under {p}: need hunk_outcomes.json "
                                    "and context_levels_slim.jsonl (or context_levels.jsonl)")
    return files

# No folder on Drive? Upload the three files straight into this Colab session instead.
# (Uploaded files are lost when the runtime restarts.)
if IN_COLAB and not Path(DATA_PATH).exists():
    from google.colab import files as colab_files
    print(f"{DATA_PATH} not found. Choose hunk_outcomes.json, context_levels_slim.jsonl and coverage.csv:")
    DATA_PATH = "/content/selector_data"
    Path(DATA_PATH).mkdir(exist_ok=True)
    for name, data in colab_files.upload().items():
        (Path(DATA_PATH) / name).write_bytes(data)

FILES = find_inputs(DATA_PATH)
for k, v in FILES.items():
    print(f"{k:9s} {v}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.metrics import roc_auc_score
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2. Configuration

Defaults follow the project documents. Change them here, not further down.

In [ ]:
LEVELS = ["H01_min", "H02_function", "H03_file", "H04_callees1", "H05_callers1",
          "H06_tests1", "H07_agent", "H08_depth2", "H09_depth2_docs", "H10_all"]

# Levels the selector may choose from. "four" is the set proposed after the 28-PR run
# (per-hunk doc Section 19.9); "two" is the H01-vs-H03 question of Section 20.
CANDIDATE_SETS = {
    "all10": LEVELS,
    "four": ["H01_min", "H03_file", "H05_callers1", "H10_all"],
    "two": ["H01_min", "H03_file"],
}
CANDIDATES = "four"

BASE_LEVEL = "H01_min"        # the cheapest level, every hunk's default
UPGRADE_LEVEL = "H03_file"    # the level the ranking selectors upgrade a hunk to
REF_LEVEL = "H03_file"        # token shares are reported against always using this level

# Severity weights (blocking, should fix, nit). "agreed" is the project's score.
WEIGHTINGS = {
    "flat": (1, 1, 1), "linear": (3, 2, 1), "agreed": (5, 2, 0.5),
    "steep": (25, 5, 1), "blocking_only": (1, 0, 0),
}
TARGET_WEIGHTING = "agreed"   # what the learned selectors are trained to raise

# Cross-validation groups: "repo" (default, the roadmap's rule) or "pr"
# ("pr" reproduces the leave-one-PR-out setting of per-hunk doc Section 20).
CV_GROUP = "repo"

# A locked test set of repositories, chosen by a seeded hash (roadmap 9.4.4).
# Off for this 15% pilot: the hash would put 72 of the 190 references in test
# (Section 3 prints which). Turn it on for the full benchmark and do not look at
# test results until every choice is frozen.
LOCK_TEST = False
TEST_REPO_FRACTION = 0.2
TEST_HASH_SALT = "cs536-selector-test"

SEEDS = [0, 1, 2]             # model seeds for the stochastic models
N_RANDOM = 1000               # random policies per budget
N_BOOT = 2000                 # bootstrap resamples (clustered by repository)
BUDGETS = [0.4, 0.5, 0.6, 0.7, 0.85]   # token shares of always REF_LEVEL
TARGET_BUDGET = 0.7           # the target of per-hunk doc Section 20.1 ...
TARGET_RECALL_SLACK = 1       # ... recall within 1 percentage point of always REF_LEVEL

USE_TFIDF = True              # TF-IDF + SVD of the H01_min text (PR text and the hunk)
TFIDF_DIM = 32
USE_EMBEDDINGS = False        # sentence embeddings of the same text (the professor's suggestion)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 32

FAST = False                  # True: fewer models and seeds, for a quick check
if FAST:
    SEEDS = [0]; N_RANDOM = 200; N_BOOT = 500

rng_global = np.random.default_rng(536)
CAND = CANDIDATE_SETS[CANDIDATES]
assert CAND[0] == BASE_LEVEL and UPGRADE_LEVEL in LEVELS and REF_LEVEL in LEVELS
print("Candidates:", CAND)

## 3. Load the outcome table and check it

`hunk_outcomes.json` holds, for each hunk and level, the input tokens, the number of
findings, and which reference comments of the PR the hunk's findings covered. The check
below rebuilds every (PR, level) union and compares it with `coverage.csv`, which the
repository's judge stage wrote.

In [ ]:
with open(FILES["outcomes"]) as f:
    OUT = json.load(f)
assert OUT["levels"] == LEVELS, OUT["levels"]
rows = OUT["rows"]
refs_by_pr = OUT["refs"]

hunks = pd.DataFrame([{k: v for k, v in r.items() if k != "levels"} for r in rows])
hunks["repo"] = hunks["pr"].str.rsplit("__", n=1).str[0]
H = len(hunks)
hunk_pos = {h: i for i, h in enumerate(hunks["hunk"])}

# Global reference index: one id per (PR, reference).
ref_rows = []
for pr, refs in refs_by_pr.items():
    for k, r in refs.items():
        ref_rows.append({"pr": pr, "ref": int(k), **r})
refs_df = pd.DataFrame(ref_rows).sort_values(["pr", "ref"]).reset_index(drop=True)
refs_df["repo"] = refs_df["pr"].str.rsplit("__", n=1).str[0]
R = len(refs_df)
ref_pos = {(p, k): i for i, (p, k) in enumerate(zip(refs_df["pr"], refs_df["ref"]))}

# COV[h, l, r]: the findings of hunk h at level l cover reference r.
# TOK[h, l]: reviewer input tokens of hunk h at level l. NFIND[h, l]: findings written.
L = len(LEVELS)
COV = np.zeros((H, L, R), dtype=bool)
TOK = np.zeros((H, L), dtype=float)
NFIND = np.zeros((H, L), dtype=float)
for r in rows:
    i = hunk_pos[r["hunk"]]
    for j, lv in enumerate(LEVELS):
        c = r["levels"][lv]
        TOK[i, j] = c["tokens"]; NFIND[i, j] = c["findings"]
        for k in c["covered"]:
            COV[i, j, ref_pos[(r["pr"], k)]] = True

# Check against coverage.csv.
cov_csv = FILES["coverage"]
if cov_csv is not None:
    cc = pd.read_csv(cov_csv)
    cc = cc[cc["covered"].astype(str) == "True"]
    want = collections.defaultdict(set)
    for e, lv, k in zip(cc["entry"], cc["level"], cc["reference"]):
        want[(e, lv)].add(int(k))
    got = collections.defaultdict(set)
    for r in rows:
        for lv in LEVELS:
            got[(r["pr"], lv)].update(r["levels"][lv]["covered"])
    bad = [k for k in set(want) | set(got) if want[k] != got[k]]
    assert not bad, f"{len(bad)} (PR, level) unions differ from coverage.csv, e.g. {bad[:3]}"
    print("coverage.csv check: every (PR, level) union matches.")
else:
    print("coverage.csv not found; the check was skipped.")

TIER_IDX = refs_df["tier"].map({"blocking": 0, "should_fix": 1, "nit": 2}).to_numpy()
assert not np.isnan(TIER_IDX.astype(float)).any(), "a reference has no severity label"
def ref_weights(name):
    return np.array(WEIGHTINGS[name], dtype=float)[TIER_IDX]
W = {name: ref_weights(name) for name in WEIGHTINGS}
IS_HUMAN = refs_df["human"].to_numpy(bool)

print(f"{hunks.pr.nunique()} PRs, {hunks.repo.nunique()} repositories, {H} hunks, {R} references")
print("Severity tiers:", refs_df["tier"].value_counts().to_dict(),
      "| written by a person:", int(IS_HUMAN.sum()))

# Which repositories a locked test set would take (printed even when it is off).
repos_sorted = sorted(hunks["repo"].unique(),
                      key=lambda s: hashlib.sha256(f"{TEST_HASH_SALT}:{s}".encode()).hexdigest())
n_test = max(1, round(TEST_REPO_FRACTION * len(repos_sorted)))
TEST_REPOS = set(repos_sorted[:n_test])
print(f"Hash-chosen test repositories ({'LOCKED, excluded below' if LOCK_TEST else 'not locked'}):",
      sorted(TEST_REPOS), "| references:", int(refs_df.repo.isin(TEST_REPOS).sum()))

## 4. Input-side features

One row per hunk. Sources:
- `hunk_outcomes.json`: size of the hunk, trace counts one hop away, hunks in the file
  and in the PR, whether the file is new;
- `context_levels.jsonl`: trace counts by hop, changed symbols by kind, each level's
  token count and item counts, whether a level equals its parent, the PR's size and
  text length, the `H01_min` text (PR title, description and the hunk's diff);
- the file path: directory depth, test path, code or not (the professor's suggestion).

`context_levels.jsonl` is about 120 MB; it is read line by line and only the fields
below are kept.

In [ ]:
CODE_EXT = {".c", ".h", ".cc", ".cpp", ".hpp", ".cxx", ".cs", ".go", ".java", ".kt", ".js",
            ".jsx", ".ts", ".tsx", ".mjs", ".py", ".rs", ".php", ".rb", ".swift", ".scala"}

hunk_feat = {}      # hunk id -> dict of features
pr_feat = {}        # pr id -> dict of features
H01_TEXT = {}       # hunk id -> H01_min text
with open(FILES["contexts"]) as f:
    for line in f:
        pr = json.loads(line)
        if pr.get("status") != "ok":
            continue
        s = pr["stats"]
        pf = {"pr_changed_files": s.get("changed_files", 0), "pr_diff_chars": s.get("diff_chars", 0),
              "pr_title_chars": s.get("pr_title_chars", 0), "pr_body_chars": s.get("pr_body_chars", 0),
              "pr_guide_files": s.get("guide_files", 0), "pr_search_hits": s.get("search_hits", 0),
              "pr_changed_symbols": s.get("changed_symbols", 0),
              "pr_graph_nodes": pr.get("graph", {}).get("nodes", s.get("graph_nodes", 0)),
              "pr_graph_edges": pr.get("graph", {}).get("edges", s.get("graph_edges", 0))}
        pr_feat[pr["id"]] = pf
        for h in pr["hunks"]:
            if h["id"] not in hunk_pos:
                continue
            hs = h["stats"]
            d = {"hunk_index": h["index"], "old_count": h.get("old_count", 0), "new_count": h.get("new_count", 0),
                 "deleted_file": int(bool(h.get("deleted_file"))), "search_names": hs.get("search_names", 0)}
            for key in ("callees_by_hop", "callers_by_hop", "test_callers_by_hop"):
                for hop in ("1", "2", "3"):
                    d[f"{key}_{hop}"] = hs.get(key, {}).get(hop, 0)
            for kind in ("Function", "Method", "Class", "Interface", "Variable", "Module", "Type"):
                d[f"sym_{kind}"] = hs.get("changed_symbols_by_label", {}).get(kind, 0)
            for lv in LEVELS:
                x = h["levels"][lv]
                d[f"tok_{lv}"] = x["input_tokens"]
                d[f"ctx_{lv}"] = x["context_tokens"]
                d[f"items_{lv}"] = x["items"]
                d[f"same_{lv}"] = int(bool(x["same_as_parent"]))
                d[f"capped_{lv}"] = int(bool(x["capped"]))
            for kind, n in h["levels"]["H10_all"]["included_by_kind"].items():
                d[f"kind10_{kind}"] = n
            hunk_feat[h["id"]] = d
            H01_TEXT[h["id"]] = h["levels"]["H01_min"]["text"]

missing = set(hunks["hunk"]) - set(hunk_feat)
assert not missing, f"{len(missing)} hunks are not in context_levels.jsonl"

F = pd.DataFrame([hunk_feat[h] for h in hunks["hunk"]]).fillna(0)
P = pd.DataFrame([pr_feat[p] for p in hunks["pr"]])
base_cols = ["pr_hunks", "file_hunks", "added", "removed", "diff_chars", "changed_symbols",
             "callees1", "callers1", "tests1", "search_hits"]
X_tab = pd.concat([hunks[base_cols].reset_index(drop=True).astype(float), F, P], axis=1)
X_tab["new_file"] = hunks["new_file"].astype(int).to_numpy()
path = hunks["file"].fillna("")
X_tab["path_depth"] = path.str.count("/").to_numpy()
X_tab["path_is_test"] = path.str.contains(r"(?i)(?:^|/)(?:tests?|spec|__tests__)(?:/|$)|_test\.|\.test\.|test_|\.spec\.").astype(int).to_numpy()
X_tab["is_code"] = hunks["ext"].isin(CODE_EXT).astype(int).to_numpy()
X_tab["hunk_rel_pos"] = X_tab["hunk_index"] / X_tab["pr_hunks"].clip(lower=1)
for lv in LEVELS[1:]:
    X_tab[f"ratio_{lv}"] = X_tab[f"tok_{lv}"] / X_tab["tok_H01_min"].clip(lower=1)
X_tab["n_distinct_inputs"] = 1 + sum(1 - X_tab[f"same_{lv}"] for lv in LEVELS[1:])
# Heavy-tailed counts on a log scale.
for c in list(X_tab.columns):
    if c.startswith(("tok_", "ctx_", "pr_", "callees", "callers", "search", "diff_chars", "kind10_", "items_")):
        X_tab[c] = np.log1p(X_tab[c].astype(float))
lang = pd.get_dummies(hunks["language"], prefix="lang").astype(int).reset_index(drop=True)
X_tab = pd.concat([X_tab, lang], axis=1).astype(float)
X_tab = X_tab.loc[:, X_tab.std() > 0]

# Leakage guard: nothing from the references, reviews or judge may be a feature.
FORBIDDEN = ("category", "context_label", "human", "tier", "covered", "finding", "matched",
             "audit", "reference", "recall", "precision", "score", "judge", "review")
leaks = [c for c in X_tab.columns if any(w in c.lower() for w in FORBIDDEN)]
assert not leaks, f"answer-side columns in the features: {leaks}"
print("Tabular features:", X_tab.shape[1])

TEXTS = [H01_TEXT[h] for h in hunks["hunk"]]
EMB = None
if USE_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        os.system("pip -q install sentence-transformers")
        from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(EMBED_MODEL)
    EMB = st.encode(TEXTS, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    print("Embeddings:", EMB.shape)

## 5. How a policy is scored

A **policy** is one level index per hunk. `evaluate` takes the union of covered
references per PR (references of different PRs never overlap, so one union over all
chosen hunks is the same) and returns recall, every weighted score, human and AI
recall, tokens and findings.

In [ ]:
LIDX = {lv: j for j, lv in enumerate(LEVELS)}
CIDX = np.array([LIDX[lv] for lv in CAND])
REF_TOKENS = TOK[:, LIDX[REF_LEVEL]].sum()

_MASKS = {}
def _subset(hunk_mask):
    """Hunk indices, reference mask and reference tokens for a set of hunks (cached)."""
    key = None if hunk_mask is None else hunk_mask.tobytes()
    if key not in _MASKS:
        idx = np.arange(H) if hunk_mask is None else np.flatnonzero(hunk_mask)
        rmask = refs_df["pr"].isin(set(hunks["pr"].iloc[idx])).to_numpy()
        _MASKS[key] = (idx, rmask, TOK[idx, LIDX[REF_LEVEL]].sum())
    return _MASKS[key]

def evaluate(choice, hunk_mask=None):
    """choice: level index (into LEVELS) per hunk. hunk_mask: hunks to include."""
    idx, rmask, ref_tok = _subset(hunk_mask)
    ch = choice[idx]
    covered = COV[idx, ch, :].any(axis=0) & rmask
    hm, am = rmask & IS_HUMAN, rmask & ~IS_HUMAN
    tok = TOK[idx, ch].sum()
    out = {"covered": int(covered.sum()), "refs": int(rmask.sum()),
           "recall": covered.sum() / rmask.sum(),
           "tokens": tok, "token_share": tok / ref_tok,
           "findings": NFIND[idx, ch].sum(),
           "recall_human": covered[hm].sum() / hm.sum() if hm.any() else np.nan,
           "recall_ai": covered[am].sum() / am.sum() if am.any() else np.nan}
    for name, w in W.items():
        out[f"score_{name}"] = (covered * w).sum() / (rmask * w).sum()
    return out

def fixed(level):
    return np.full(H, LIDX[level])

# Per-hunk gain at each level: the weighted references the hunk's own findings cover.
# A reference covered by two hunks counts in both, so this is a training target only;
# every reported number uses the exact union above.
def hunk_gain(weighting):
    return (COV * W[weighting][None, None, :]).sum(axis=2)
GAIN = hunk_gain(TARGET_WEIGHTING)       # (H, L)
GAIN_FLAT = hunk_gain("flat")

USE = ~hunks["repo"].isin(TEST_REPOS).to_numpy() if LOCK_TEST else np.ones(H, bool)
print(f"Hunks used for development: {USE.sum()} of {H}")

fixed_rows = [{"policy": f"always {lv}", **evaluate(fixed(lv), USE)} for lv in LEVELS]
FIXED = pd.DataFrame(fixed_rows)
show_cols = ["policy", "covered", "refs", "recall", "score_agreed", "recall_human", "recall_ai",
             "token_share", "findings"]
FIXED[show_cols].round(3)

Check: with every hunk on the dev set, the covered counts above must equal the
per-hunk doc, Section 19.1 (106, 106, 116, 111, 109, 107, 99, 96, 99, 94 of 190).

In [ ]:
if not LOCK_TEST:
    expected = [106, 106, 116, 111, 109, 107, 99, 96, 99, 94]
    assert FIXED["covered"].tolist() == expected, FIXED["covered"].tolist()
    print("Matches Section 19.1.")

# Where the level can matter at all.
cov_any = COV.any(axis=2)
changes = np.array([len({tuple(COV[i, j].nonzero()[0]) for j in range(L)}) > 1 for i in range(H)])
print(f"Hunks with no covered reference at any level: {(~cov_any.any(axis=1)).sum()}")
print(f"Hunks whose covered references change with the level: {changes.sum()}")
b, u = LIDX[BASE_LEVEL], LIDX[UPGRADE_LEVEL]
print(f"{UPGRADE_LEVEL} gains over {BASE_LEVEL} on {(GAIN_FLAT[:, u] > GAIN_FLAT[:, b]).sum()} hunks, "
      f"loses on {(GAIN_FLAT[:, u] < GAIN_FLAT[:, b]).sum()}")

## 6. Folds

Leave one group out: with 20 repositories, each fold holds out one repository.
All predictions below are **out of fold**: a hunk's prediction comes from a model that
never saw its repository (or its PR, with `CV_GROUP = "pr"`).

In [ ]:
GROUPS = (hunks["repo"] if CV_GROUP == "repo" else hunks["pr"]).to_numpy()
dev_idx = np.flatnonzero(USE)
FOLDS = [(dev_idx[tr], dev_idx[te]) for tr, te in LeaveOneGroupOut().split(dev_idx, groups=GROUPS[dev_idx])]
print(f"{len(FOLDS)} folds by {CV_GROUP}")

## 7. Feature sets

- `tabular`: the input-side features of Section 4.
- `tabular+text`: plus TF-IDF of the `H01_min` text reduced with SVD, fitted inside
  each fold on the training hunks only.
- `text`: the text alone.
- `language`: language one-hot only. A shortcut check: if this does as well as the
  full set, the selector is learning the repository's language, not the change.
- `tabular+embed`: with sentence embeddings, if `USE_EMBEDDINGS`.

In [ ]:
LANG_COLS = [c for c in X_tab.columns if c.startswith("lang_")]
XT = X_tab.to_numpy()
XL = X_tab[LANG_COLS].to_numpy()

def text_block(tr, te, seed=0):
    vec = TfidfVectorizer(token_pattern=r"[A-Za-z_][A-Za-z0-9_]+", lowercase=True,
                          min_df=2, max_features=20000, sublinear_tf=True)
    A = vec.fit_transform([TEXTS[i] for i in tr])
    B = vec.transform([TEXTS[i] for i in te])
    k = min(TFIDF_DIM, A.shape[1] - 1, len(tr) - 1)
    svd = TruncatedSVD(n_components=k, random_state=seed).fit(A)
    return svd.transform(A), svd.transform(B)

def emb_block(tr, te, seed=0):
    k = min(EMBED_DIM, len(tr) - 1)
    pca = PCA(n_components=k, random_state=seed).fit(EMB[tr])
    return pca.transform(EMB[tr]), pca.transform(EMB[te])

_X_CACHE = {}
def make_X(fs, tr, te, seed=0):
    """Feature matrices for one fold. Text and embedding reductions are fitted on the
    training hunks only. Cached per (feature set, fold): they do not depend on the model."""
    key = (fs, tr.tobytes(), te.tobytes())
    if key not in _X_CACHE:
        _X_CACHE[key] = _make_X(fs, tr, te)
    return _X_CACHE[key]

def _make_X(fs, tr, te, seed=0):
    parts_tr, parts_te = [], []
    if fs in ("tabular", "tabular+text", "tabular+embed"):
        parts_tr.append(XT[tr]); parts_te.append(XT[te])
    if fs == "language":
        parts_tr.append(XL[tr]); parts_te.append(XL[te])
    if fs in ("tabular+text", "text"):
        a, bb = text_block(tr, te, seed); parts_tr.append(a); parts_te.append(bb)
    if fs == "tabular+embed":
        a, bb = emb_block(tr, te, seed); parts_tr.append(a); parts_te.append(bb)
    A, B = np.hstack(parts_tr), np.hstack(parts_te)
    sc = StandardScaler().fit(A)
    return np.nan_to_num(sc.transform(A)), np.nan_to_num(sc.transform(B))

FEATURE_SETS = ["tabular", "language"] + (["tabular+text", "text"] if USE_TFIDF else []) \
    + (["tabular+embed"] if USE_EMBEDDINGS else [])
print("Feature sets:", FEATURE_SETS)

## 8. Baselines

All by lookup, on the development hunks:
- **fixed levels** (above) and the **best fixed level chosen inside each fold** on its
  training repositories;
- **random level** per hunk from the candidate set;
- **random upgrade at a budget**: hunks in random order get `UPGRADE_LEVEL` until the
  token budget is used, the rest stay at `BASE_LEVEL` (per-hunk doc Section 20.3);
- **cheapest first**, a rule with no learning: the hunks whose upgrade costs the fewest
  tokens get it first;
- **hindsight**: the best level per hunk knowing the outcome, an upper bound that also
  contains run-to-run noise. It is not a method.

In [ ]:
def budget_tokens(share):
    return share * TOK[USE, LIDX[REF_LEVEL]].sum()

def at_budget(frontier, share, by="tokens"):
    """The frontier point used at a budget. by="tokens": the point that spends the most
    tokens within the budget, which needs no outcome (used for every selector).
    by="score": the best-scoring point within it (only for the hindsight upper bound)."""
    ok = frontier[frontier["token_share"] <= share + 1e-9]
    if not len(ok):
        return None
    if by == "score":
        return ok.sort_values(["score_agreed", "covered"], ascending=False).iloc[0]
    return ok.sort_values("token_share", ascending=False, kind="stable").iloc[0]

def prefix_eval(order, k, up=UPGRADE_LEVEL):
    ch = fixed(BASE_LEVEL).copy(); ch[order[:k]] = LIDX[up]
    return evaluate(ch, USE)

def budget_prefix(order, share, up=UPGRADE_LEVEL):
    extra = TOK[order, LIDX[up]] - TOK[order, LIDX[BASE_LEVEL]]
    base_tok = TOK[USE, LIDX[BASE_LEVEL]].sum()
    k = int(np.searchsorted(np.cumsum(extra), budget_tokens(share) - base_tok, side="right"))
    return prefix_eval(order, k, up)

# Random upgrade at each budget.
rand_rows = []
for share in BUDGETS:
    vals = []
    for s in range(N_RANDOM):
        order = np.random.default_rng(s).permutation(dev_idx)
        vals.append(budget_prefix(order, share))
    v = pd.DataFrame(vals)
    rand_rows.append({"budget": share, "covered_mean": v.covered.mean(),
                      "covered_p05": v.covered.quantile(0.05), "covered_p95": v.covered.quantile(0.95),
                      "score_mean": v.score_agreed.mean(), "score_p05": v.score_agreed.quantile(0.05),
                      "score_p95": v.score_agreed.quantile(0.95)})
RANDOM_BUDGET = pd.DataFrame(rand_rows)

# Random level from the candidate set.
rl = pd.DataFrame([evaluate(np.random.default_rng(s).choice(CIDX, size=H), USE) for s in range(N_RANDOM)])

# Cheapest first.
extra_cost = TOK[:, LIDX[UPGRADE_LEVEL]] - TOK[:, LIDX[BASE_LEVEL]]
cheap_order = dev_idx[np.argsort(extra_cost[dev_idx], kind="stable")]
CHEAP = pd.DataFrame([{"budget": s, **budget_prefix(cheap_order, s)} for s in BUDGETS])

# Best fixed level chosen in-fold.
bf_choice = np.zeros(H, int)
for tr, te in FOLDS:
    m = np.zeros(H, bool); m[tr] = True
    best = max(CAND, key=lambda lv: (evaluate(fixed(lv), m)["score_" + TARGET_WEIGHTING], -LIDX[lv]))
    bf_choice[te] = LIDX[best]
BEST_FIXED_CV = evaluate(bf_choice, USE)

# Hindsight: per hunk, the candidate level with the largest gain minus lambda * tokens.
def lambda_policy(pred, lam):
    """pred: (H, len(CAND)) predicted gain. Ties go to the cheaper (earlier) level."""
    util = pred - lam * TOK[:, CIDX] / 1000.0
    return CIDX[np.argmax(util, axis=1)]          # argmax takes the first, i.e. cheaper, level

LAMBDAS = np.concatenate([[0.0], np.logspace(-5, 1.5, 60), [1e6]])
def lambda_frontier(pred):
    return pd.DataFrame([{"lam": lam, **evaluate(lambda_policy(pred, lam), USE)} for lam in LAMBDAS])

HINDSIGHT = lambda_frontier(GAIN[:, CIDX])

base_row = {"policy": f"random level from {CANDIDATES}", **rl.mean(numeric_only=True).to_dict()}
summary_baselines = pd.concat([FIXED, pd.DataFrame([
    {"policy": "best fixed level, chosen in-fold", **BEST_FIXED_CV}, base_row])], ignore_index=True)
display_cols = ["policy", "covered", "recall", "score_agreed", "recall_human", "recall_ai", "token_share"]
print(summary_baselines[display_cols].round(3).to_string(index=False))
print("\nRandom upgrade to", UPGRADE_LEVEL, "at a budget (share of always", REF_LEVEL, "tokens):")
print(RANDOM_BUDGET.round(3).to_string(index=False))
print("\nCheapest first:")
print(CHEAP[["budget", "covered", "score_agreed", "token_share"]].round(3).to_string(index=False))

# Check against per-hunk doc Section 20.3 (same setting: H01_min -> H03_file, all 28 PRs):
# cheapest first covers 108, 111, 114, 117, 117 and random upgrades 108.6 ... 114.8 on average.
if not LOCK_TEST and (BASE_LEVEL, UPGRADE_LEVEL, REF_LEVEL) == ("H01_min", "H03_file", "H03_file") \
        and BUDGETS == [0.4, 0.5, 0.6, 0.7, 0.85]:
    assert CHEAP["covered"].tolist() == [108, 111, 114, 117, 117], CHEAP["covered"].tolist()
    print("Cheapest first matches Section 20.3. Random means there: 108.6, 110.3, 111.8, 113.1, 114.8; here:",
          RANDOM_BUDGET["covered_mean"].round(1).tolist())

## 9. Selector A: per-level gain regression (the roadmap's preferred target)

One regression model on stacked rows (hunk, candidate level). Inputs: the hunk's
features, the level as a one-hot, and that level's own token, item and same-as-parent
features. Target: the weighted references the hunk's findings cover at that level.
At selection time each hunk takes the level with the highest predicted gain minus
λ × tokens; sweeping λ draws the quality-cost frontier without retraining.

In [ ]:
LEVEL_KINDS = ("tok", "ctx", "items", "same", "ratio")   # the same columns for every level

def level_block(lv, idx):
    cols = []
    for kind in LEVEL_KINDS:
        c = f"{kind}_{lv}"
        cols.append(X_tab[c].to_numpy()[idx] if c in X_tab.columns else np.zeros(len(idx)))
    return np.column_stack(cols)

LB_SCALE = StandardScaler().fit(np.vstack([level_block(lv, np.arange(H)) for lv in CAND]))

def stacked(X_hunk, idx):
    """Rows (hunk, candidate level), level-major: hunk features + level one-hot + the
    level's own token, item, same-as-parent and ratio features."""
    blocks = []
    for k, lv in enumerate(CAND):
        oh = np.zeros((len(idx), len(CAND))); oh[:, k] = 1
        blocks.append(np.hstack([X_hunk, oh, LB_SCALE.transform(level_block(lv, idx))]))
    return np.vstack(blocks)

def regressors(seed):
    return {
        "ridge": lambda: Ridge(alpha=10.0),
        "random_forest": lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                       max_features=0.3, random_state=seed, n_jobs=-1),
        "boosted_trees": lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
                                                               max_iter=200, min_samples_leaf=20,
                                                               l2_regularization=1.0, random_state=seed),
        "mlp": lambda: MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-2, max_iter=800,
                                    early_stopping=True, random_state=seed),
    }

def oof_regression(fs, model_name, seed):
    pred = np.zeros((H, len(CAND)))
    for tr, te in FOLDS:
        A, B = make_X(fs, tr, te, seed)
        y = GAIN[tr][:, CIDX].T.reshape(-1)            # level-major, same order as stacked()
        m = regressors(seed)[model_name]()
        m.fit(stacked(A, tr), y)
        pred[te] = m.predict(stacked(B, te)).reshape(len(CAND), len(te)).T
    return pred

t0 = time.time()
REG = {}            # (fs, model, seed) -> OOF predicted gain (H, len(CAND))
for fs in FEATURE_SETS:
    for mn in ["ridge", "random_forest", "boosted_trees", "mlp"]:
        for seed in (SEEDS if mn != "ridge" else SEEDS[:1]):
            REG[(fs, mn, seed)] = oof_regression(fs, mn, seed)
    print(f"{fs}: done ({time.time() - t0:.0f} s)")

## 10. Selector B: rank hunks for an upgrade (binary, per-hunk doc Section 20)

Label: the hunk covers more weighted references at `UPGRADE_LEVEL` than at
`BASE_LEVEL`. A classifier ranks the hunks, and hunks are upgraded in that order until
the budget is used. The ranking score (ROC AUC, 0.5 is a coin flip) is reported with a
bootstrap interval over repositories.

In [ ]:
Y_UP = (GAIN[:, LIDX[UPGRADE_LEVEL]] > GAIN[:, LIDX[BASE_LEVEL]]).astype(int)
print(f"Hunks where {UPGRADE_LEVEL} adds weighted references over {BASE_LEVEL} (net): "
      f"{Y_UP[USE].sum()} of {USE.sum()}")

def classifiers(seed):
    return {
        "logistic": lambda: LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
        "random_forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.3,
                                                        class_weight="balanced", random_state=seed, n_jobs=-1),
        "boosted_trees": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                                min_samples_leaf=20, l2_regularization=1.0,
                                                                class_weight="balanced", random_state=seed),
        "mlp": lambda: MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-1, max_iter=800,
                                     early_stopping=True, random_state=seed),
    }

def oof_binary(fs, model_name, seed):
    p = np.full(H, np.nan)
    for tr, te in FOLDS:
        A, B = make_X(fs, tr, te, seed)
        if Y_UP[tr].min() == Y_UP[tr].max():
            p[te] = Y_UP[tr].mean(); continue
        m = classifiers(seed)[model_name]()
        m.fit(A, Y_UP[tr])
        p[te] = m.predict_proba(B)[:, 1]
    return p

def cluster_boot_auc(y, s, groups, n=N_BOOT, seed=0):
    g = np.unique(groups); rng = np.random.default_rng(seed); vals = []
    by = {k: np.flatnonzero(groups == k) for k in g}
    for _ in range(n):
        idx = np.concatenate([by[k] for k in rng.choice(g, size=len(g))])
        if y[idx].min() != y[idx].max():
            vals.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

t0 = time.time()
BIN = {}
for fs in FEATURE_SETS:
    for mn in ["logistic", "random_forest", "boosted_trees", "mlp"]:
        for seed in (SEEDS if mn != "logistic" else SEEDS[:1]):
            BIN[(fs, mn, seed)] = oof_binary(fs, mn, seed)
print(f"done ({time.time() - t0:.0f} s)")

repo_groups = hunks["repo"].to_numpy()
auc_rows = []
for (fs, mn, seed), p in BIN.items():
    auc_rows.append({"features": fs, "model": mn, "seed": seed,
                     "auc": roc_auc_score(Y_UP[USE], p[USE])})
AUC = pd.DataFrame(auc_rows).groupby(["features", "model"]).agg(
    auc_mean=("auc", "mean"), auc_sd=("auc", "std"), seeds=("seed", "count")).reset_index()
ci = []
for _, r in AUC.iterrows():
    p = np.mean([BIN[(r.features, r.model, s)] for s in SEEDS if (r.features, r.model, s) in BIN], axis=0)
    lo, hi = cluster_boot_auc(Y_UP[USE], p[USE], repo_groups[USE])
    ci.append((lo, hi))
AUC["ci_low"], AUC["ci_high"] = zip(*ci)
print(AUC.sort_values("auc_mean", ascending=False).round(3).to_string(index=False))

## 11. Selector C: direct classification of the hindsight level

The label is the hindsight level at a fixed λ (ties to the cheaper level); a
multi-class classifier predicts it. Each λ gives one policy, so this draws a few
points on the frontier. Most hunks have no reference, so the label is mostly the
cheapest level.

In [ ]:
# Gains are in severity weight (0 to ~10 per hunk) and costs in thousands of tokens,
# so useful λ values lie around 0.01 to 1.
LAMBDA_LABELS = [0.01, 0.03, 0.1, 0.3, 1.0]

def oof_direct(fs, model_name, seed, lam):
    y = lambda_policy(GAIN[:, CIDX], lam)                      # level index per hunk
    yk = np.searchsorted(CIDX, y)                              # class index into CAND
    ch = np.full(H, CIDX[0])
    for tr, te in FOLDS:
        A, B = make_X(fs, tr, te, seed)
        if len(np.unique(yk[tr])) < 2:
            ch[te] = CIDX[yk[tr][0]]; continue
        m = classifiers(seed)[model_name]()
        m.fit(A, yk[tr])
        ch[te] = CIDX[m.classes_[np.argmax(m.predict_proba(B), axis=1)]]
    return ch

t0 = time.time()
DIRECT_ROWS = []
for fs in [f for f in FEATURE_SETS if f != "text"]:
    for mn in ["logistic", "boosted_trees"]:
        for lam in LAMBDA_LABELS:
            for seed in SEEDS[:1]:                     # one seed: this family is the slowest
                ch = oof_direct(fs, mn, seed, lam)
                DIRECT_ROWS.append({"features": fs, "model": mn, "label_lambda": lam, "seed": seed,
                                    **evaluate(ch, USE)})
DIRECT = pd.DataFrame(DIRECT_ROWS)
print(f"done ({time.time() - t0:.0f} s)")
for lam in LAMBDA_LABELS:
    lab = pd.Series(lambda_policy(GAIN[:, CIDX], lam)[USE]).map(dict(enumerate(LEVELS))).value_counts()
    print(f"Hindsight label distribution at λ = {lam}:", lab.to_dict())

## 12. Results at fixed token budgets

Every policy at each budget (share of the tokens of always `REF_LEVEL`): the best point
of its frontier that fits the budget. Learned rows are the mean over seeds (with the
standard deviation). The **target** is per-hunk doc Section 20.1: recall within one
reference of always `REF_LEVEL` at 70% or less of its tokens. It must also be beaten
by more than the random spread (5th to 95th percentile).

In [ ]:
def frontier_rows(name, fs, mn, fr_list):
    out = []
    for share in BUDGETS:
        pts = [at_budget(fr, share) for fr in fr_list]
        pts = [p for p in pts if p is not None]
        if not pts:
            continue
        d = pd.DataFrame(pts)
        out.append({"selector": name, "features": fs, "model": mn, "budget": share,
                    "covered": d.covered.mean(), "covered_sd": d.covered.std(ddof=0),
                    "score_agreed": d.score_agreed.mean(), "score_sd": d.score_agreed.std(ddof=0),
                    "recall_human": d.recall_human.mean(), "recall_ai": d.recall_ai.mean(),
                    "token_share": d.token_share.mean()})
    return out

FRONTIERS = {}
rows_b = []
for fs in FEATURE_SETS:
    for mn in ["ridge", "random_forest", "boosted_trees", "mlp"]:
        frs = [lambda_frontier(REG[(fs, mn, s)]) for s in SEEDS if (fs, mn, s) in REG]
        FRONTIERS[("A gain regression", fs, mn)] = frs
        rows_b += frontier_rows("A gain regression", fs, mn, frs)
    for mn in ["logistic", "random_forest", "boosted_trees", "mlp"]:
        frs = []
        for s in SEEDS:
            if (fs, mn, s) not in BIN:
                continue
            order = dev_idx[np.argsort(-BIN[(fs, mn, s)][dev_idx], kind="stable")]
            frs.append(pd.DataFrame([budget_prefix(order, b) for b in BUDGETS]))
        FRONTIERS[("B upgrade ranking", fs, mn)] = frs
        rows_b += frontier_rows("B upgrade ranking", fs, mn, frs)
for share in BUDGETS:
    p = at_budget(HINDSIGHT, share, by="score")
    rows_b.append({"selector": "hindsight (upper bound)", "features": "-", "model": "-", "budget": share,
                   "covered": p.covered, "score_agreed": p.score_agreed, "token_share": p.token_share,
                   "recall_human": p.recall_human, "recall_ai": p.recall_ai})
for _, r in CHEAP.iterrows():
    rows_b.append({"selector": "cheapest first (no learning)", "features": "-", "model": "-",
                   "budget": r.budget, "covered": r.covered, "score_agreed": r.score_agreed,
                   "token_share": r.token_share, "recall_human": r.recall_human, "recall_ai": r.recall_ai})
BUDGET_TABLE = pd.DataFrame(rows_b).merge(
    RANDOM_BUDGET[["budget", "covered_mean", "covered_p95", "score_mean", "score_p95"]], on="budget")
BUDGET_TABLE["beats_random_p95"] = BUDGET_TABLE["score_agreed"] > BUDGET_TABLE["score_p95"]

fx = FIXED.set_index("policy").loc[f"always {REF_LEVEL}"]
ref_cov, ref_score, n_refs = int(fx["covered"]), fx["score_agreed"], int(fx["refs"])
# Target: recall at most TARGET_RECALL_SLACK percentage points below always REF_LEVEL.
need = math.ceil(ref_cov - TARGET_RECALL_SLACK / 100 * n_refs - 1e-9)
BUDGET_TABLE["meets_target"] = (BUDGET_TABLE["covered"] >= need) & (BUDGET_TABLE["budget"] <= TARGET_BUDGET)
t = BUDGET_TABLE[BUDGET_TABLE.budget == TARGET_BUDGET].copy()
print(f"Target: {need} or more of {n_refs} references at {TARGET_BUDGET:.0%} of the tokens or less")
print(f"Always {REF_LEVEL}: {ref_cov} references, score {ref_score:.3f}")
print(f"Random upgrade at {TARGET_BUDGET:.0%}: mean {RANDOM_BUDGET.set_index('budget').loc[TARGET_BUDGET, 'covered_mean']:.1f} "
      f"references, 95th percentile score {RANDOM_BUDGET.set_index('budget').loc[TARGET_BUDGET, 'score_p95']:.3f}")
cols = ["selector", "features", "model", "covered", "covered_sd", "score_agreed", "score_sd",
        "recall_human", "recall_ai", "token_share", "beats_random_p95", "meets_target"]
print(t.sort_values("score_agreed", ascending=False)[cols].round(3).head(25).to_string(index=False))

Direct classification (Selector C): each λ is one policy; mean over seeds.

In [ ]:
dsum = DIRECT.groupby(["features", "model", "label_lambda"]).agg(
    covered=("covered", "mean"), score_agreed=("score_agreed", "mean"),
    token_share=("token_share", "mean"), recall_human=("recall_human", "mean")).reset_index()
print(dsum.round(3).to_string(index=False))

## 13. Quality-cost frontier

Score (left) and references covered (right) against token share. Grey band: random
upgrades (5th to 95th percentile). Dots: fixed levels. The best learned frontier per
selector family, by score at `TARGET_BUDGET`, is drawn.

In [ ]:
def best_config(family):
    t = BUDGET_TABLE[(BUDGET_TABLE.selector == family) & (BUDGET_TABLE.budget == TARGET_BUDGET)]
    r = t.sort_values("score_agreed", ascending=False).iloc[0]
    return (family, r.features, r.model)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, rcol in [(axes[0], "score_agreed", "score"), (axes[1], "covered", "covered")]:
    lo = RANDOM_BUDGET[f"{rcol}_p05"]; hi = RANDOM_BUDGET[f"{rcol}_p95"]
    ax.fill_between(RANDOM_BUDGET.budget, lo, hi, color="0.85", label="random upgrade, 5-95%")
    ax.plot(RANDOM_BUDGET.budget, RANDOM_BUDGET[f"{rcol}_mean"], color="0.5", lw=1)
    for _, r in FIXED.iterrows():
        ax.scatter(r.token_share, r[metric], color="black", s=18, zorder=3)
        ax.annotate(r.policy.replace("always ", "")[:3], (r.token_share, r[metric]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
    h = HINDSIGHT.sort_values("token_share")
    ax.plot(h.token_share, h[metric], ls="--", color="tab:green", label="hindsight (upper bound)")
    for fam, color in [("A gain regression", "tab:blue"), ("B upgrade ranking", "tab:orange")]:
        key = best_config(fam)
        fr = FRONTIERS[key][0].sort_values("token_share")
        ax.plot(fr.token_share, fr[metric], color=color, marker=".", ms=3,
                label=f"{fam}: {key[1]}, {key[2]}")
    ax.plot(CHEAP.budget, CHEAP[metric], color="tab:red", marker="x", label="cheapest first")
    ax.set_xscale("log"); ax.set_xlabel(f"input tokens, share of always {REF_LEVEL} (log)")
    ax.set_ylabel("severity-weighted score" if metric == "score_agreed" else "references covered, of 190")
    ax.axvline(TARGET_BUDGET, color="0.6", ls=":")
axes[0].legend(fontsize=7, loc="lower right")
fig.suptitle(f"Per-hunk selector, out of fold by {CV_GROUP}; candidates: {CANDIDATES}; one reviewer run")
fig.tight_layout()
fig.savefig(OUT_DIR / "frontier.png", dpi=160)
plt.show()

## 14. Is the best selector better than the baselines? (paired, clustered bootstrap)

For the best configuration of each family at `TARGET_BUDGET` (picked on these same
out-of-fold results, so the comparison is **optimistic**: many configurations were
tried), the difference in score against always `REF_LEVEL` and against a random
upgrade at the same budget, resampling repositories.

In [ ]:
def policy_at_budget(key, share, seed_i=0):
    fam, fs, mn = key
    s = SEEDS[seed_i]
    if fam == "A gain regression":
        pred = REG[(fs, mn, s)] if (fs, mn, s) in REG else REG[(fs, mn, SEEDS[0])]
        best = None                                  # the λ that spends the most tokens within budget
        for lam in LAMBDAS:
            ch = lambda_policy(pred, lam); e = evaluate(ch, USE)
            if e["token_share"] <= share + 1e-9 and (best is None or e["token_share"] > best[1]):
                best = (ch, e["token_share"])
        return best[0]
    p = BIN[(fs, mn, s)] if (fs, mn, s) in BIN else BIN[(fs, mn, SEEDS[0])]
    order = dev_idx[np.argsort(-p[dev_idx], kind="stable")]
    extra = TOK[order, LIDX[UPGRADE_LEVEL]] - TOK[order, LIDX[BASE_LEVEL]]
    k = int(np.searchsorted(np.cumsum(extra), budget_tokens(share) - TOK[USE, LIDX[BASE_LEVEL]].sum(), side="right"))
    ch = fixed(BASE_LEVEL).copy(); ch[order[:k]] = LIDX[UPGRADE_LEVEL]
    return ch

def boot_diff(ch_a, ch_b, n=N_BOOT, seed=0):
    """Score of policy a minus policy b, resampling repositories with replacement."""
    repos = hunks["repo"].to_numpy(); g = np.unique(repos[USE]); rng = np.random.default_rng(seed)
    def covered_by(ch, r):
        idx = np.flatnonzero((repos == r) & USE)
        return COV[idx, ch[idx], :].any(0)
    cov_a = {r: covered_by(ch_a, r) for r in g}
    cov_b = {r: covered_by(ch_b, r) for r in g}
    rmask = {r: (refs_df["repo"] == r).to_numpy() for r in g}
    w = W["agreed"]
    def score(sample, cov):
        num = sum((cov[r] & rmask[r]) @ w for r in sample); den = sum(rmask[r] @ w for r in sample)
        return num / den
    obs = score(g, cov_a) - score(g, cov_b)
    vals = [score(s, cov_a) - score(s, cov_b) for s in (rng.choice(g, size=len(g)) for _ in range(n))]
    return obs, *np.percentile(vals, [2.5, 97.5])

cmp_rows = []
for fam in ["A gain regression", "B upgrade ranking"]:
    key = best_config(fam)
    ch = policy_at_budget(key, TARGET_BUDGET)
    for other_name, other in [(f"always {REF_LEVEL}", fixed(REF_LEVEL)), (f"always {BASE_LEVEL}", fixed(BASE_LEVEL)),
                              ("cheapest first", None)]:
        if other is None:
            extra = TOK[cheap_order, LIDX[UPGRADE_LEVEL]] - TOK[cheap_order, LIDX[BASE_LEVEL]]
            k = int(np.searchsorted(np.cumsum(extra), budget_tokens(TARGET_BUDGET) - TOK[USE, LIDX[BASE_LEVEL]].sum(), side="right"))
            other = fixed(BASE_LEVEL).copy(); other[cheap_order[:k]] = LIDX[UPGRADE_LEVEL]
        d, lo, hi = boot_diff(ch, other)
        cmp_rows.append({"selector": f"{fam}: {key[1]}, {key[2]}", "versus": other_name,
                         "score_diff": d, "ci_low": lo, "ci_high": hi, "token_share": evaluate(ch, USE)["token_share"]})
COMPARE = pd.DataFrame(cmp_rows)
print(COMPARE.round(3).to_string(index=False))
print("\nNote: the random-upgrade baseline at this budget scores",
      f"{RANDOM_BUDGET.set_index('budget').loc[TARGET_BUDGET, 'score_mean']:.3f} on average "
      f"(95th percentile {RANDOM_BUDGET.set_index('budget').loc[TARGET_BUDGET, 'score_p95']:.3f}).")

## 15. Groups: human and AI references, tiers, context labels, categories

The professor asked for results split by who wrote the reference. The answer-side
fields are used **here only, to group the evaluation**, never as features.

In [ ]:
def group_table(policies):
    out = []
    for name, ch in policies.items():
        covered = COV[np.flatnonzero(USE), ch[USE], :].any(0)
        rmask = refs_df["pr"].isin(set(hunks["pr"][USE])).to_numpy()
        row = {"policy": name}
        for col in ["human", "tier", "context", "category"]:
            for val, g in refs_df[rmask].groupby(col):
                row[f"{col}={val}"] = f"{covered[g.index].sum()}/{len(g)}"
        out.append(row)
    return pd.DataFrame(out).set_index("policy").T

policies = {f"always {lv}": fixed(lv) for lv in CAND}
for fam in ["A gain regression", "B upgrade ranking"]:
    policies[f"{fam} @ {TARGET_BUDGET:.0%}"] = policy_at_budget(best_config(fam), TARGET_BUDGET)
GROUPS_TABLE = group_table(policies)
GROUPS_TABLE

## 16. Which features the ranking model uses

Permutation importance of the best Selector B configuration, measured out of fold on
each held-out repository and averaged. It is noisy with 20 small folds; read the top of
the list only.

In [ ]:
from sklearn.inspection import permutation_importance
key = best_config("B upgrade ranking"); fs, mn = key[1], key[2]
if fs == "tabular":
    imp = np.zeros(XT.shape[1]); n = 0
    for tr, te in FOLDS:
        if Y_UP[te].min() == Y_UP[te].max() or Y_UP[tr].min() == Y_UP[tr].max():
            continue
        A, B = make_X(fs, tr, te)
        m = classifiers(SEEDS[0])[mn]().fit(A, Y_UP[tr])
        r = permutation_importance(m, B, Y_UP[te], scoring="roc_auc", n_repeats=10, random_state=0)
        imp += r.importances_mean; n += 1
    IMPORTANCE = pd.Series(imp / max(n, 1), index=X_tab.columns).sort_values(ascending=False)
    print(f"{mn}, {n} folds with both classes:")
    print(IMPORTANCE.head(15).round(4).to_string())
else:
    print(f"The best ranking configuration uses '{fs}'; importance is shown for tabular features only.")

## 17. Save everything

CSVs, the frontier figure, out-of-fold predictions, and a manifest with the data hashes
and the settings, so a result can be traced back to its inputs.

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FIXED.to_csv(OUT_DIR / "fixed_levels.csv", index=False)
summary_baselines.to_csv(OUT_DIR / "baselines.csv", index=False)
RANDOM_BUDGET.to_csv(OUT_DIR / "random_upgrade_by_budget.csv", index=False)
BUDGET_TABLE.to_csv(OUT_DIR / "budget_table.csv", index=False)
AUC.to_csv(OUT_DIR / "upgrade_auc.csv", index=False)
DIRECT.to_csv(OUT_DIR / "direct_classification.csv", index=False)
COMPARE.to_csv(OUT_DIR / "bootstrap_comparisons.csv", index=False)
GROUPS_TABLE.to_csv(OUT_DIR / "groups.csv")
HINDSIGHT.to_csv(OUT_DIR / "hindsight_frontier.csv", index=False)
oof = hunks[["hunk", "pr", "repo", "language", "file"]].copy()
oof["y_upgrade_helps"] = Y_UP
for (fs, mn, s), p in BIN.items():
    oof[f"p_up|{fs}|{mn}|{s}"] = p
oof.to_csv(OUT_DIR / "oof_upgrade_probabilities.csv", index=False)

manifest = {
    "written_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    "inputs": {k: str(v) for k, v in FILES.items()},
    "hunk_outcomes_sha256": sha256(FILES["outcomes"]),
    "contexts_sha256": sha256(FILES["contexts"]),
    "hunks": int(H), "prs": int(hunks.pr.nunique()), "repos": int(hunks.repo.nunique()), "references": int(R),
    "dev_hunks": int(USE.sum()), "lock_test": LOCK_TEST, "test_repos": sorted(TEST_REPOS) if LOCK_TEST else [],
    "candidates": CAND, "base_level": BASE_LEVEL, "upgrade_level": UPGRADE_LEVEL, "ref_level": REF_LEVEL,
    "target_weighting": TARGET_WEIGHTING, "weightings": WEIGHTINGS, "cv_group": CV_GROUP, "folds": len(FOLDS),
    "seeds": SEEDS, "n_random": N_RANDOM, "n_boot": N_BOOT, "budgets": BUDGETS,
    "feature_sets": FEATURE_SETS, "n_tabular_features": int(X_tab.shape[1]),
    "tfidf_dim": TFIDF_DIM if USE_TFIDF else None, "embed_model": EMBED_MODEL if USE_EMBEDDINGS else None,
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
                 "python": sys.version.split()[0]},
    "note": "Pilot on 15% of AACR-Bench (one reviewer run). Not a final result.",
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1))
print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name)